# Day 22: Deep Learning Mini-Project & Checkpoint

Today **you** build and train a small neural network, alone, from start to end.

**Checkpoint:** at the end of today you can say:
> "I understand what this code is doing, and I can train a small neural network."

| Part | What you do | Time |
|---|---|---|
| A | Read code and explain each line | 25 min |
| B | Mini-project: predict exam results | 75 min |
| C | Experiments: change settings, compare | 25 min |
| D | Answer 6 short questions | 15 min |
| ✅ | Checkpoint list | 5 min |

**Rules:** You can look at your old notebooks. Ask the teacher only after you try for 5 minutes.

In [25]:
%pip install torch

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\New folder\\Lib\\site-packages\\torch\\_inductor\\cpu_vec_isa.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/124.1 MB 1.9 MB/s eta 0:01:07
    --------------------------------------- 2.1/124.1 MB 3.8 MB/s eta 0:00:33
   - -------------------------------------- 3.9/124.1 MB 5.3 MB/s eta 0:00:23
   - -------------------------------------- 5.8/124.1 MB 6.1 MB/s eta 0:00:20
   - -------------------------------------- 6.0/124.1 MB 6.3 MB/s eta 0:00:19
   -- ------------------------------------- 8.7/124.1 MB 5.9 MB/s eta 0:00:20
   ---- ----------------------------------- 13.1/124.1 MB 7.9 MB/s eta 0:00:15
   ---- ----------------------------------- 14.2/124.1 MB 7.9 MB/s eta 0:00:14
   ---- ----------------------------------- 15.5/124.1 MB 7.4 MB/s eta 0:00:15
   ----- ---------------------------------- 16.8/124.1 MB 7.8 MB/s eta 0:00:14
   


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import numpy as np                                     # arrays of numbers
import pandas as pd                                    # tables (read CSV files)
import matplotlib.pyplot as plt                        # charts
import torch                                           # PyTorch: tensors + training
from torch import nn                                   # layers, ReLU, loss functions
from sklearn.model_selection import train_test_split   # split into train / validation / test
from sklearn.preprocessing import StandardScaler       # make columns the same scale

---
# Part A: Read the code and explain it

This code trains a small network on **loans.csv**: will the bank **approve** the loan (1) or not (0)?

**Your task:** run it. Then, in the next cell, write a short comment **after every line**: what does this line do?

In [ ]:
loans = pd.read_csv("loans.csv")
X = loans[["monthly_income", "loan_amount"]].values
y = loans[["approved"]].values

X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    guess = model(X)
    loss = loss_fn(guess, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("final loss:", round(loss.item(), 3))

In [ ]:
# Copy the code here and write a comment after EVERY line.
# Example:  loans = pd.read_csv("loans.csv")    # read the table from the file

loans = pd.read_csv("loans.csv") # reading the csv file
X = loans[["monthly_income", "loan_amount"]].values    # selecting the features and converting them to a numpy array
y = loans[["approved"]].values     # selecting the target variable and converting it to a numpy array 
  
X = StandardScaler().fit_transform(X) #scaling the features
X = torch.tensor(X, dtype=torch.float32)  # converting the X features to a PyTorch tensor of type float32
y = torch.tensor(y, dtype=torch.float32) # converting the y target to a PyTorch tensor of type float32

torch.manual_seed(42)  # setting the random seed for reproducibility
model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid()) # defining a sequential model with 2 input features, 8 hidden units, ReLU activation, 1 output unit, and Sigmoid activation
loss_fn = nn.BCELoss() # defining the binary cross-entropy loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.01) # defining the Adam optimizer

for epoch in range(100): # training the model for 100 epochs
    guess = model(X)        # making predictions using the model
    loss = loss_fn(guess, y) # calculating the loss between the predictions and the true labels
    optimizer.zero_grad()       # resetting the gradients of the model parameters to zero
    loss.backward()         # computing the gradients of the loss with respect to the model parameters
    optimizer.step()        # updating the model parameters using the computed gradients

print("final loss:", round(loss.item(), 3))     # printing the final loss after training




**✏️ Questions about Part A** (write your answer in the cell below each one)

1. Why does this model end with `nn.Sigmoid()`?
2. Why do we use `BCELoss` here?
3. In Part B we predict **3** results. Which last layer and which loss will we use there?

*Your answers:*

1.
2.
3.

---
# Part B: Mini-project: predict exam results

A college in Lahore wants to know early which students need help.
For **400 students** they have 5 numbers and the final **result**.

| Column | Meaning |
|---|---|
| `study_hours` | hours of study per day (0 to 6) |
| `attendance_pct` | attendance in % (40 to 100) |
| `assignments_done` | assignments done, out of 10 |
| `sleep_hours` | hours of sleep per night |
| `phone_hours` | hours on the phone per day |
| `result` | **0 = Fail, 1 = Pass, 2 = Distinction** |

**Goal:** train a network that guesses `result`. Then check it on the test students.

These 2 classes are the same as in class. **Run this cell, do not change it.**

In [34]:
class Net(nn.Module):
    def __init__(self, n_inputs, n_hidden, n_outputs):
        super().__init__()                          # start the nn.Module part
        self.hidden = nn.Linear(n_inputs, n_hidden) # input → hidden (weights + biases)
        self.relu = nn.ReLU()                       # activation: negative → 0
        self.output = nn.Linear(n_hidden, n_outputs)# hidden → output

    def forward(self, x):
        return self.output(self.relu(self.hidden(x)))


class Trainer:
    def __init__(self, model, loss_fn, lr):
        self.model = model
        self.loss_fn = loss_fn
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.train_losses, self.val_losses = [], []

    def fit(self, X_train, y_train, X_val, y_val, epochs):
        for epoch in range(epochs):
            loss = self.loss_fn(self.model(X_train), y_train)   # 1. guess + 2. how wrong
            self.optimizer.zero_grad()                          # 3. clear old blame
            loss.backward()                                     # 4. find blame (backprop)
            self.optimizer.step()                               # 5. fix (gradient descent)
            self.train_losses.append(loss.item())
            with torch.no_grad():                               # mock exam: check only
                self.val_losses.append(self.loss_fn(self.model(X_val), y_val).item())

    def predict(self, X):
        with torch.no_grad():
            return self.model(X)

    def plot(self):
        plt.plot(self.train_losses, label="train (homework)")
        plt.plot(self.val_losses, label="validation (mock exam)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()

## Step 1: Load and look

- Read `exam_results.csv` into `exams`.
- Print the first 5 rows and the shape.
- Count how many students are in each result with `.value_counts()`.

In [3]:
# your code here

import pandas as pd

In [11]:
exam = pd.read_csv("exam_results.csv")  
print(exam.head(5))
print(exam.shape)
print(exam["result"].value_counts())

   study_hours  attendance_pct  assignments_done  sleep_hours  phone_hours  \
0          3.8              60                 6          8.3          3.5   
1          5.4              96                10          8.4          2.0   
2          4.7              49                 6          7.1          5.7   
3          1.4              71                 1          6.6          5.8   
4          1.8              45                10          5.0          6.6   

   result  
0       1  
1       2  
2       1  
3       0  
4       0  
(400, 6)
result
1    180
0    140
2     80
Name: count, dtype: int64


## Step 2: X and y

- `X` = the 5 input columns (everything **except** `result`). Hint: `exams.drop(columns="result")`
- `y` = the `result` column.
- Use `.values` on both.

In [13]:
# your code here
X = exam.drop(columns=["result"]).values
y = exam[["result"]].values

## Step 3: Split 60 / 20 / 20

Homework (train), mock exam (validation), final exam (test). Use `random_state=42` both times.

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split( X, y, test_size=0.4, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

## Step 4: Scale

- `attendance_pct` is 40 to 100, `sleep_hours` is 4 to 9. Big numbers would push the weights too much.
- Make a `StandardScaler`. **Fit it on train only**, then transform train, val and test.
- Why train only? The test students are the "final exam". We must not peek at them.

In [20]:
# your code here
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## Step 5: Tensors

- Inputs → `torch.float32` (decimals).
- Answers → `torch.long` (whole numbers), because `CrossEntropyLoss` wants the class number 0, 1, 2.

In [29]:
# your code here

import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

C:\Users\DELL\AppData\Local\Temp\ipykernel_15812\694488829.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\DELL\AppData\Local\Temp\ipykernel_15812\694488829.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)
C:\Users\DELL\AppData\Local\Temp\ipykernel_15812\694488829.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
C:\Users\DELL\AppData\Local\Temp\ipykernel_15812\694488829

## Step 6: Build and train

- Model: `Net` with **5 inputs, 16 hidden, 3 outputs**. Write `torch.manual_seed(42)` first.
- Loss: `nn.CrossEntropyLoss()`. Learning rate `0.01`.
- Train **100 epochs** with `Trainer`, then `.plot()` the losses.

In [37]:
torch.manual_seed(42)

model = Net(5, 16, 3)

loss_fn = nn.CrossEntropyLoss()

trainer = Trainer(model, loss_fn, lr=0.01)

trainer.fit(
    X_train, y_train,
    X_val, y_val,
    epochs=100
)

trainer.plot()

RuntimeError: 0D or 1D target tensor expected, multi-target not supported

**✏️ Look at your plot:** do both lines go down? Does the validation line stop going down or go up at the end?

*Your answer:*

## Step 7: Evaluate

- Guess for the **train** and **test** students: `trainer.predict(...)`, then `.argmax(dim=1)`.
- Accuracy = correct ÷ total. Print both.
- Make a table of real vs guessed with `pd.crosstab(y_test, test_guess)`.

In [ ]:
# your code here

**✏️ Questions:**
1. Is test accuracy lower than train accuracy? What does that mean?
2. In the table, which mistakes happen most? (Fail ↔ Pass, Pass ↔ Distinction, or Fail ↔ Distinction?)

*Your answers:*

1.
2.

## Step 8: Predict for new students

| Name | study | attendance | assignments | sleep | phone |
|---|---|---|---|---|---|
| Ali | 5.0 | 90 | 9 | 7.5 | 1.0 |
| Sara | 1.0 | 55 | 2 | 5.0 | 6.0 |

- Put them in a table (2 rows × 5 numbers).
- **Scale with the same `scaler`** (only `.transform`).
- Tensor → `trainer.predict` → `torch.softmax(..., dim=1)` to see the chances.

In [ ]:
# your code here

---
# Part C: Experiments

One setting at a time. Run `run(...)` and **write the numbers in the table**.

In [ ]:
def run(n_hidden=16, lr=0.01, epochs=100):
    torch.manual_seed(42)
    model = Net(5, n_hidden, 3)
    trainer = Trainer(model, nn.CrossEntropyLoss(), lr=lr)
    trainer.fit(X_train_t, y_train_t, X_val_t, y_val_t, epochs=epochs)
    train_acc = (trainer.predict(X_train_t).argmax(dim=1).numpy() == y_train).mean()
    test_acc = (trainer.predict(X_test_t).argmax(dim=1).numpy() == y_test).mean()
    print(f"hidden={n_hidden}, lr={lr}, epochs={epochs} → train {train_acc:.1%}, test {test_acc:.1%}")

run()   # our model from Part B

In [ ]:
# Try these one by one:
# run(lr=0.001)
# run(lr=0.1)
# run(n_hidden=4)
# run(n_hidden=64)
# run(epochs=20)
# run(epochs=300)

| Change | Train acc | Test acc | What happened? |
|---|---|---|---|
| normal (16, 0.01, 100) | | | |
| lr = 0.001 | | | |
| lr = 0.1 | | | |
| hidden = 4 | | | |
| hidden = 64 | | | |
| epochs = 20 | | | |
| epochs = 300 | | | |

---
# Part D: Short questions

Answer in 1–2 lines each.

1. What does **one epoch** mean?
2. What do `loss.backward()` and `optimizer.step()` do?
3. Why do we split off **test** data and never train on it?
4. Why did we **scale** the columns?
5. Why is there **no softmax layer** in our `Net`?
6. Train 96%, test 82%. What is the name for this gap, and give one way to make it smaller.

*Your answers:*

1.
2.
3.
4.
5.
6.

---
# ✅ Checkpoint

Tick each line only if it is true. Show the teacher.

- [ ] I can explain what every line of the Part A code does.
- [ ] I can load a CSV and make X and y.
- [ ] I can split into train / validation / test and scale with train only.
- [ ] I can make tensors (float32 for inputs, long for classes).
- [ ] I can build a `Net`, train it with `Trainer`, and read the loss plot.
- [ ] I can work out accuracy on test data.
- [ ] I know when to use Sigmoid + BCELoss and when to use CrossEntropyLoss.
- [ ] I can predict for a new person (scale first!).

---
# 🏠 Problem

Same steps, **yes/no** problem: will the bank approve the loan? Use `loans.csv`.

1. X = `monthly_income`, `loan_amount`, `years_in_job`, `has_defaulted_before`. y = `approved`.
2. Split 60 / 20 / 20, scale with train only.
3. Tensors: X **float32**, y **float32** with shape (n, 1). (BCELoss wants decimals.)
4. Model: `nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())`, loss `nn.BCELoss()`, 100 epochs.
5. Guess = chance > 0.5. Print the test accuracy.
6. Write 2 lines: what changed compared with the exam project, and why?